# EDA de datos sucios — tickets de soporte IT/SaaS

Explora `data/raw/messy_it_tickets.csv`: un dataset sintético de 10,000 tickets de soporte generado por `src/generators/dirty_data_generator.py` con los defectos típicos de datos reales de operaciones TI — fechas en formatos/timezones mezclados, JSON anidado de profundidad variable, nombres de empresa con typos y duplicados, montos en distintos formatos de moneda, y valores faltantes representados de formas inconsistentes.

El objetivo es **diagnosticar** estos problemas antes de limpiarlos — el pipeline de limpieza (`src/pipeline.py`) resuelve cada uno de los hallazgos de este notebook.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

COLOR_PRIMARY = "#2a78d6"
COLOR_SECONDARY = "#eb6834"

df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "messy_it_tickets.csv")
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()


## 1. Filas completamente vacías

Simulan exports corruptos: ninguna columna tiene valor. El pipeline las descarta antes de cualquier otro procesamiento (`df.dropna(how="all")`).


In [ ]:
blank_rows = df[df.isna().all(axis=1)]
print(f"Filas completamente vacías: {len(blank_rows)} ({len(blank_rows) / len(df):.2%})")


## 2. Valores faltantes por columna

Incluye tanto `NaN` reconocido automáticamente por pandas (`""`, `"N/A"`, `"null"`, etc.) como placeholders que sobreviven como texto literal (`"-"`, `"?"`) y que el pipeline debe manejar explícitamente.


In [ ]:
null_counts = df.isna().sum().sort_values(ascending=False)
print(null_counts)

fig, ax = plt.subplots(figsize=(7, 4))
null_counts[null_counts > 0].plot(kind="barh", ax=ax, color=COLOR_PRIMARY)
ax.set_xlabel("Valores faltantes (NaN reconocidos por pandas)")
ax.set_title("Valores faltantes por columna")
plt.tight_layout()
plt.show()


## 3. Fechas: formatos y timezones mezclados

`created_at` y `resolved_at` combinan al menos 6 formatos distintos (ISO, `dd/mm/aaaa`, `mm/dd/aaaa` 12h, nombre de mes, etc.) con sufijos de timezone que van desde un offset numérico hasta abreviaciones (`EST`, `PST`, `CET`, `UTC`, `Z`).


In [ ]:
print("Ejemplos de created_at tal como llegan:")
for value in df["created_at"].dropna().sample(10, random_state=42):
    print(f"  {value!r}")


## 4. `user_metadata`: JSON anidado de profundidad variable

Desde `"{}"`/`null` hasta objetos con 3 niveles de anidamiento y listas — no se puede asumir una estructura fija.


In [ ]:
print("Ejemplos de user_metadata:")
for value in df["user_metadata"].dropna().sample(6, random_state=1):
    print(f"  {value}")


## 5. Nombres de empresa: typos y duplicados

La misma empresa aparece con mayúsculas/minúsculas distintas, sufijos legales (`Inc.`, `LLC`), espacios dobles, y typos de un solo carácter.


In [ ]:
company_counts = df["company_name"].value_counts()
print(f"Valores únicos de company_name: {df['company_name'].nunique()} (para ~20 empresas reales)")
print("\nEjemplo — variantes que probablemente son 'Nova Systems':")
print([c for c in company_counts.index if isinstance(c, str) and "ova" in c.lower() and "ystem" in c.lower()])


## 6. Costos: formatos de moneda mezclados

Símbolo antes o después del monto, coma o punto como separador decimal, código de moneda como sufijo, y algunos reembolsos negativos.


In [ ]:
print("Ejemplos de cost:")
for value in df["cost"].dropna().sample(10, random_state=7):
    print(f"  {value!r}")


## 7. Conclusiones — qué debe resolver el pipeline

- **Filas vacías** -> descartar (`pipeline.run_pipeline`, primer paso).
- **JSON anidado variable** -> `json_normalizer.normalize_json_column` aplana a columnas planas, tolerando profundidad y nulos.
- **Fechas heterogéneas** -> `datetime_cleaner.parse_to_utc` + `to_iso8601` normalizan todo a UTC en ISO 8601 (con la limitación honesta de que día/mes ambiguo sin metadata de locale no tiene solución perfecta).
- **Nombres con typos/duplicados** -> `string_cleaner.unify_similar_names` (rapidfuzz) colapsa variantes a un nombre canónico.
- **Monedas mezcladas** -> `string_cleaner.parse_currency` detecta el separador decimal correcto por formato.
- **Faltantes remanentes** -> `missing_data_imputer` (media condicional por categoría para `cost`, interpolación temporal para `response_time_hours`); lo que no se puede imputar con confianza queda expuesto por `validators.schema_validator`, que separa filas válidas de inválidas en vez de forzar un valor inventado.
